# Add and remove EPA SWMM LID controls

This notebook demonstrates the `swmmx` LID control API added in version `0.0.41`.

Public endpoints:

```python
m.add.hydrology.lid_control(...)
m.remove.hydrology.lid_control(ids, force=False)
```

A SWMM LID control is a multi-row `[LID_CONTROLS]` object. It has one type row and one or more process-layer rows such as `SURFACE`, `SOIL`, `STORAGE`, `PAVEMENT`, `DRAIN`, `DRAINMAT`, and `REMOVALS`.

In [ ]:
from swmmx import swmm

m = swmm(new="SI")
print(m.get.lid_control.count())

## Supported EPA SWMM LID types

`type=` accepts friendly names or SWMM codes:

| Friendly name | SWMM code | Typical required layers |
|---|---:|---|
| `bio_retention_cell` | `BC` | surface, soil, storage |
| `rain_garden` | `RG` | surface, soil |
| `green_roof` | `GR` | surface, soil, drain_mat |
| `infiltration_trench` | `IT` | surface, storage |
| `permeable_pavement` | `PP` | surface, pavement, storage |
| `rain_barrel` | `RB` | storage, drain |
| `rooftop_disconnection` | `RD` | surface |
| `vegetative_swale` | `VS` | surface |

Optional layers depend on LID type. `strict_layers=True` is the default and catches invalid combinations early.

## Bio-retention cell with surface, soil, storage, and drain layers

In [ ]:
m.add.hydrology.lid_control(
    "BR1",
    type="bio_retention_cell",
    surface={
        "storage_depth": 150,
        "vegetation_fraction": 0.10,
        "roughness": 0.05,
        "slope": 0.0,
        "side_slope": 0.0,
    },
    soil={
        "thickness": 450,
        "porosity": 0.45,
        "field_capacity": 0.20,
        "wilting_point": 0.10,
        "conductivity": 25,
        "conductivity_slope": 10,
        "suction_head": 75,
    },
    storage={
        "height": 300,
        "void_ratio": 0.75,
        "seepage_rate": 5,
        "clogging_factor": 0,
    },
    drain={
        "coefficient": 0.5,
        "exponent": 0.5,
        "offset_height": 0,
        "delay": 0,
        "open_level": 0,
        "closed_level": 0,
    },
)

m.section("LID_CONTROLS")

## Query LID control and layer parameters

Layer rows are also available through the existing `m.get` style.

In [ ]:
print(m.get.lid_control.count())
print(m.get.lid_control.type("BR1"))
print(m.get.lid_surface.storage_depth("BR1"))
print(m.get.lid_soil.porosity("BR1"))
print(m.get.lid_storage.height("BR1"))
print(m.get.lid_drain.coefficient("BR1"))

## Add all EPA SWMM LID types

Empty layer dictionaries are accepted and filled with SWMM-safe zeros. This makes it easy to create a skeleton and refine values later.

In [ ]:
m.add.hydrology.lid_control("RG1", "rain_garden", surface={}, soil={})
m.add.hydrology.lid_control("GR1", "green_roof", surface={}, soil={}, drain_mat={"thickness": 50, "void_ratio": 0.5, "roughness": 0.1})
m.add.hydrology.lid_control("IT1", "infiltration_trench", surface={}, storage={})
m.add.hydrology.lid_control("PP1", "permeable_pavement", surface={}, pavement={}, storage={})
m.add.hydrology.lid_control("RB1", "rain_barrel", storage={}, drain={})
m.add.hydrology.lid_control("RD1", "rooftop_disconnection", surface={})
m.add.hydrology.lid_control("VS1", "vegetative_swale", surface={})

print(m.get.lid_control.count())
print(m.get.lid_control.type(["BR1", "RG1", "GR1", "IT1", "PP1", "RB1", "RD1", "VS1"]))

## Pavement, storage, drain mat, and pollutant removal parameters

The add API supports the extended SWMM 5.2 fields:

- pavement: `regeneration_interval`, `regeneration_fraction`
- storage: `covered`
- drain: `open_level`, `closed_level`, `control_curve`
- drain mat: `thickness`, `void_ratio`, `roughness`
- removals: pollutant/removal pairs

In [ ]:
m.add.hydrology.lid_control(
    "PP2",
    type="PP",
    surface={"storage_depth": 5},
    pavement={
        "thickness": 100,
        "void_ratio": 0.15,
        "impervious_surface_fraction": 0,
        "permeability": 100,
        "clogging_factor": 0,
        "regeneration_interval": 30,
        "regeneration_fraction": 0.25,
    },
    storage={"height": 200, "void_ratio": 0.75, "seepage_rate": 2, "clogging_factor": 0, "covered": 0},
    drain={"coefficient": 0.2, "exponent": 0.5, "offset_height": 0, "delay": 0},
    removals={"TSS": 80, "TN": 25},
)

[row for row in m.section("LID_CONTROLS") if row[0] == "PP2"]

## Validation examples

Missing required layers and invalid layer combinations raise clear SWMMX errors.

In [ ]:
from swmmx import MissingRequiredParameterError, InvalidParameterError

try:
    m.add.hydrology.lid_control("BAD_GR", type="green_roof", surface={}, soil={})
except MissingRequiredParameterError as err:
    print(err)

try:
    m.add.hydrology.lid_control("BAD_RB", type="rain_barrel", surface={}, storage={}, drain={})
except InvalidParameterError as err:
    print(err)

## Remove LID controls

Removing a control deletes all of its `[LID_CONTROLS]` rows. If a control is referenced by `[LID_USAGE]`, removal blocks by default. Use `force=True` only when you intentionally want to delete dependent usage rows.

In [ ]:
# Safe remove: no LID_USAGE dependency
m.remove.hydrology.lid_control("VS1")
print(m.get.lid_control.count())

In [ ]:
# Demonstrate dependency protection with a simple usage row.
m._document.append_row("LID_USAGE", ["S1", "BR1", 1, 100, 10, 0, 100, 0])

try:
    m.remove.hydrology.lid_control("BR1")
except Exception as err:
    print(type(err).__name__, err)

summary = m.remove.hydrology.lid_control("BR1", force=True)
print(summary)
print(m.section("LID_USAGE"))

## Save the edited model

After adding/removing LID controls, `m.modified` is `True`. If the model had simulation results, SWMMX marks them stale until the model is run again.

In [ ]:
print(m.modified)
# m.save("model_with_lids.inp")